# fair-prep — Colab / Local runner

End-to-end pipeline: SFT, DPO, cross-size LoRA transfer, merging scaling-law.

**Where it runs**:
- **Colab** (T4/A100/L4): bootstrap cell clones repo + installs unsloth.
- **Local VS Code** (Mac MPS / Linux CUDA): bootstrap detects existing checkout, skips clone. Pick the `.venv` kernel from VS Code top-right.

Device auto-detected by `lb` (CUDA > MPS > CPU).

## 1 · Bootstrap (Colab clone + install, or local detect)

In [ ]:
import os, sys, pathlib

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
print('IN_COLAB =', IN_COLAB)

if IN_COLAB:
    os.environ.setdefault('WITH_UNSLOTH', '1')
    # os.environ['HF_TOKEN'] = 'hf_xxx'  # uncomment for gated models
else:
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'lora-bench' / 'lb').exists():
            os.chdir(p / 'lora-bench')
            break
    else:
        raise RuntimeError('lora-bench/ not found above ' + str(here))
    print('local cwd:', os.getcwd())

In [ ]:
# Colab only: run bootstrap (clone repo + uv + deps + unsloth). Output streams here.
if IN_COLAB:
    !curl -sL https://raw.githubusercontent.com/Shumatsurontek/fair-prep/main/colab/setup.sh | bash
    %cd /content/fair-prep/lora-bench

import torch
print('cwd        =', os.getcwd())
print('torch      =', torch.__version__)
print('cuda avail =', torch.cuda.is_available())
print('mps  avail =', torch.backends.mps.is_available())

In [ ]:
!./lb status
!./lb models

N = 200 if torch.cuda.is_available() else 20
# IPython `!` interpolates {N} from notebook scope.
!./lb eval gsm8k   -M qwen3-0.6b -n {N} -B 16 -o results/baseline.json
!./lb eval math500 -M qwen3-0.6b -n {N} -B 16 -o results/math500_baseline.json

In [ ]:
N = 200 if torch.cuda.is_available() else 20
!./lb eval gsm8k   -M qwen3-0.6b -n {N} -o results/baseline.json
!./lb eval math500 -M qwen3-0.6b -n {N} -o results/math500_baseline.json

## 3 · SFT (standard, TRL)

In [ ]:
# Small `--max-steps 50 -n 200` for smoke run on local MPS.
# Remove for a full run on CUDA.
STEPS = -1 if torch.cuda.is_available() else 50
TRAIN_N = '' if torch.cuda.is_available() else '-n 200'
!./lb train sft -M qwen3-0.6b -c configs/sft_gsm8k.yaml -s {STEPS} {TRAIN_N}
!./lb eval gsm8k   -a runs/qwen3_06b_sft_gsm8k/final -n {N} -o results/sft.json
!./lb eval math500 -a runs/qwen3_06b_sft_gsm8k/final -n {N} -o results/math500_sft.json

## 4 · SFT (unsloth fast path — CUDA only)

Auto-falls back to TRL if no CUDA or unsloth not installed.

In [ ]:
!./lb train sft -M qwen3-0.6b -c configs/sft_gsm8k.yaml --fast -s {STEPS} {TRAIN_N}

## 5 · Cross-size LoRA transfer (research axis)

Teacher (Qwen3-4B) LoRA → student (Qwen3-0.6B) via OLS projection on calibration activations. Layer mapping via linear stride.

In [ ]:
# 5.1 download public Qwen3-4B math LoRA (adapter only, ~120 MB)
from huggingface_hub import snapshot_download
snapshot_download(
    'saaduddinM/MATH_SFT_Q3.0-4BI_r16',
    local_dir='runs/teacher_4b/math_sft',
    allow_patterns=['adapter_*', '*.json'],
)

In [ ]:
# 5.2 build calibration set from MATH-500 prompts
import json
from datasets import load_dataset
os.makedirs('data', exist_ok=True)
n_calib = 256 if torch.cuda.is_available() else 32
ds = load_dataset('HuggingFaceH4/MATH-500', split='test').shuffle(seed=42).select(range(n_calib))
with open('data/calib_math.jsonl', 'w') as f:
    for ex in ds:
        f.write(json.dumps({'text': ex['problem']}) + '\n')
print(f'wrote {n_calib} calib prompts')

In [ ]:
# 5.3 project teacher → student. `-d` omitted → lb auto-picks cuda/mps/cpu.
!./lb transfer \
    -T Qwen/Qwen3-4B-Instruct-2507 \
    -S Qwen/Qwen3-0.6B \
    -a runs/teacher_4b/math_sft \
    -C data/calib_math.jsonl \
    -o runs/transferred/math_sft \
    -n {n_calib}

In [ ]:
# 5.4 eval transferred adapter
!./lb eval math500 -a runs/transferred/math_sft -n {N} -o results/math500_transferred.json

## 6 · Merging scaling law (arXiv:2509.24244)

After training k teacher experts (one per math domain), project them all, merge, eval, fit `L(k) = L∞ + A/(k+b)`. Run only when ≥ 2 transferred adapters exist.

In [ ]:
import glob
transferred = sorted(glob.glob('runs/transferred/*/adapter_model.safetensors'))
transferred = [p.rsplit('/', 1)[0] for p in transferred]
print('available transferred adapters:', transferred)
if len(transferred) >= 2:
    flags = ' '.join(f'-a {p}' for p in transferred[:3])
    !./lb merge -m ties {flags} -o runs/merged_ties
    !./lb eval math500 -a runs/merged_ties -n {N} -o results/scaling/ties.json
else:
    print('skip: need ≥ 2 transferred adapters. Train more teachers in cell 5.')

## 7 · Report

In [ ]:
!./lb report
from IPython.display import Markdown
Markdown(open('results/REPORT.md').read())